## load data

In [1]:
import numpy as np
import pandas as pd
from sympy import limit
import wandb

api = wandb.Api(
    api_key="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",
    timeout=60
)

# 1. Get last 10 runs (sorted by creation time descending)
runs = api.runs(
    "eibl-usc/graph-clip",
    filters={
        # "display_name": {"$regex": "trained_on_.._eval_on_.._..?_shot_.+"},
        # "config.dataset": "ukr_rus_twitter",
        # "config.task_name": "neighbor_matching",
    },
    order="-created_at",
    per_page=10,
    # limit to 10:
    lazy=False
)

In [8]:
rows = []
for run in runs[:1000]:
    attrs = getattr(run, "_attrs", {}) or {}
    params = ((attrs.get("config") or {}).get("params") or {})
    summary = attrs.get("summaryMetrics") or {}

    rows.append({
        "run_id": attrs.get("name"),
        "display_name": attrs.get("displayName"),
        "state": attrs.get("state"),
        "dataset": params.get("dataset"),
        "task_name": params.get("task_name"),
        "prefix": params.get("prefix"),
        "pretrained_model_run": params.get("pretrained_model_run"),
        "n_shots": params.get("n_shots"),
        "n_way": params.get("n_way"),
        "n_query": params.get("n_query"),
        "zero_shot": params.get("zero_shot"),
        "test_accuracy": summary.get("test_accuracy"),
        "train_accuracy": summary.get("train_accuracy"),
        "test_f1": summary.get("test_f1"),
        "test_roc_auc": summary.get("test_roc_auc"),
        "created_at": attrs.get("createdAt"),
        'steps': attrs['historyKeys']['keys'].get('_step', {}).get('typeCounts', [{}])[0].get('count', np.nan)
    })
df = pd.DataFrame(rows)
df["train1_dataset"] = df["pretrained_model_run"].str.extract(r"train1_(ukr_rus_twitter|midterm|covid19_twitter)_")
df["train1_task"] = df["pretrained_model_run"].str.extract(r"train1_.+?_(nm|pl|lp)_")
df["eval1_task"] = df["task_name"].map({
    "neighbor_matching": "nm",
    "temporal_link_prediction": "lp",
    "classification": "pl",
})
df['created_at'] = pd.to_datetime(df['created_at'])
df["shot_label"] = df.apply(lambda r: 0 if bool(r.get("zero_shot", False)) else r["n_shots"], axis=1)
# df['is_eval'] = df['display_name'].str.contains(r"eval")
# plot_df = df[df["eval1_task"].isin(EVAL_TASKS) & df["train1_task"].eq("nm")].copy()
df['eval1_dataset'] = df['dataset']
df['trained_on_display_name'] = df.pretrained_model_run.str.split('/').str[1]
df['month/day'] = df['created_at'].dt.month.astype(str) + '/' + df['created_at'].dt.day.astype(str)
df = df.sort_values('created_at', ascending=False)
d = {
    'trained_on_n_way': 'n_way',
    'trained_on_n_query': 'n_query',
    'trained_on_n_shots': 'n_shots',
    'trained_on_steps': 'steps',
}
mask = df['trained_on_display_name'].isin(df['display_name'])
existing_trained_on_display_names = df.trained_on_display_name[mask]
dname2stats = df.set_index('display_name').loc[existing_trained_on_display_names][list(d.values())]
for col, stat in d.items():
    df[col] = df.trained_on_display_name.map(dname2stats[stat].to_dict())
df__ = df.copy()

# df = df[df.state.ne('running')]
# allowed_shots = [0, 1, 2, 5, 10]
# df = df[df["n_shots"].isin(allowed_shots)]
unique_on = ['train1_dataset', 'train1_task', 'eval1_dataset', 'eval1_task', 'n_shots']#, 'n_query', 'n_way']
df = df.sort_values('created_at', ascending=False)
df = df.drop_duplicates(subset=unique_on, keep='first')
df_ = df.copy()

## plots

In [11]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

TRAIN_DATASET = 'covid19_twitter'
EVAL_DATASET = 'covid19_twitter'
EVAL_TASKS = ["nm", "lp", "pl"]
colors = {"nm": "#00b4d8", "lp": "#f72585", "pl": "#80b918"}

i_cross = df.train1_dataset.eq(TRAIN_DATASET) & df.eval1_dataset.eq(EVAL_DATASET)
plot_df_cross = df[i_cross].copy()
i_same = (df.train1_dataset.eq(EVAL_DATASET) & df.eval1_dataset.eq(EVAL_DATASET)
          & df.prefix.str.startswith("trained"))
plot_df_same = df[i_same].copy()

fig = make_subplots(rows=1, cols=len(EVAL_TASKS), shared_yaxes=True,
                    subplot_titles=[f"Eval: {t.upper()}" for t in EVAL_TASKS])

seen = set()  # dedupe legend across subplots
for col, eval_task in enumerate(EVAL_TASKS, start=1):
    # Cross (solid)
    sub_c = plot_df_cross[plot_df_cross["eval1_task"] == eval_task]
    for tt in sorted(sub_c["train1_task"].dropna().unique()):
        s = sub_c[sub_c["train1_task"] == tt].sort_values("shot_label").dropna(subset=["test_roc_auc"])
        s.loc[s.test_roc_auc.lt(0.5), 'test_roc_auc'] = 0.5 + (0.5-s.test_roc_auc)
        if not s.pretrained_model_run.duplicated(keep=False).all():
            print(f"Warning: pretrained_model_run is not unique for train1_task={tt} and eval1_task={eval_task}")
        name = tt.upper()
        fig.add_trace(go.Scatter(
            x=s["shot_label"], y=s["test_roc_auc"],
            mode="lines+markers", name=name,
            legendgroup=name, showlegend=name not in seen,
            line=dict(color=colors.get(tt)),
            customdata=s[["display_name", "pretrained_model_run", 'steps']].values,
            hovertemplate="<b>%{customdata[1]}</b><br>shots=%{x}<br>AUC=%{y:.3f}<br>train_steps=%{customdata[2]}<br>%{customdata[0]}<extra></extra>",
        ), row=1, col=col)
        seen.add(name)

    # Same-dataset baseline (dashed)
    sub_s = plot_df_same[plot_df_same["eval1_task"] == eval_task]
    for tt in sorted(sub_s["train1_task"].dropna().unique()):
        s = sub_s[sub_s["train1_task"] == tt].sort_values("shot_label").dropna(subset=["test_roc_auc"])
        if not s.pretrained_model_run.duplicated(keep=False).all():
            print(f"Warning: pretrained_model_run is not unique for train1_task={tt} and eval1_task={eval_task}")
        name = f"{tt.upper()} (baseline)"
        fig.add_trace(go.Scatter(
            x=s["shot_label"], y=s["test_roc_auc"],
            mode="lines+markers", name=name,
            legendgroup=name, showlegend=name not in seen,
            line=dict(color=colors.get(tt), dash="dash"), opacity=0.5,
            customdata=s[["display_name", "pretrained_model_run", 'steps']].values,
            hovertemplate="<b>%{customdata[1]}</b><br>shots=%{x}<br>AUC=%{y:.3f}<br>train_steps=%{customdata[2]}<br>%{customdata[0]}<extra></extra>",
        ), row=1, col=col)
        seen.add(name)

    fig.update_xaxes(title_text="Shots", row=1, col=col)

fig.update_yaxes(title_text="Test ROC-AUC", row=1, col=1)
fig.update_layout(
    title=f"{TRAIN_DATASET} → {EVAL_DATASET}",
    height=450, width=1100,
    legend=dict(orientation="h", y=-0.2, x=0.5, xanchor="center"),
    hovermode="closest",
)
fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

TRAIN_DATASET = 'ukr_rus_twitter'
EVAL_DATASET = 'ukr_rus_twitter'
EVAL_TASKS = ["nm", "lp", "pl"]
colors = {"nm": "#00b4d8", "lp": "#f72585", "pl": "#80b918"}

i_cross = df.train1_dataset.eq(TRAIN_DATASET) & df.eval1_dataset.eq(EVAL_DATASET)
plot_df_cross = df[i_cross].copy()
i_same = (df.train1_dataset.eq(EVAL_DATASET) & df.eval1_dataset.eq(EVAL_DATASET)
          & df.prefix.str.startswith("trained"))
plot_df_same = df[i_same].copy()

fig = make_subplots(rows=1, cols=len(EVAL_TASKS), shared_yaxes=True,
                    subplot_titles=[f"Eval: {t.upper()}" for t in EVAL_TASKS])

seen = set()  # dedupe legend across subplots
for col, eval_task in enumerate(EVAL_TASKS, start=1):
    # Cross (solid)
    sub_c = plot_df_cross[plot_df_cross["eval1_task"] == eval_task]
    for tt in sorted(sub_c["train1_task"].dropna().unique()):
        s = sub_c[sub_c["train1_task"] == tt].sort_values("shot_label").dropna(subset=["test_roc_auc"])
        s.loc[s.test_roc_auc.lt(0.5), 'test_roc_auc'] = 0.5 + (0.5-s.test_roc_auc)
        if not s.pretrained_model_run.duplicated(keep=False).all():
            print(f"Warning: pretrained_model_run is not unique for train1_task={tt} and eval1_task={eval_task}")
        name = tt.upper()
        fig.add_trace(go.Scatter(
            x=s["shot_label"], y=s["test_roc_auc"],
            mode="lines+markers", name=name,
            legendgroup=name, showlegend=name not in seen,
            line=dict(color=colors.get(tt)),
            customdata=s[["display_name", "pretrained_model_run", 'steps']].values,
            hovertemplate="<b>%{customdata[1]}</b><br>shots=%{x}<br>AUC=%{y:.3f}<br>train_steps=%{customdata[2]}<br>%{customdata[0]}<extra></extra>",
        ), row=1, col=col)
        seen.add(name)

    # Same-dataset baseline (dashed)
    sub_s = plot_df_same[plot_df_same["eval1_task"] == eval_task]
    for tt in sorted(sub_s["train1_task"].dropna().unique()):
        s = sub_s[sub_s["train1_task"] == tt].sort_values("shot_label").dropna(subset=["test_roc_auc"])
        if not s.pretrained_model_run.duplicated(keep=False).all():
            print(f"Warning: pretrained_model_run is not unique for train1_task={tt} and eval1_task={eval_task}")
        name = f"{tt.upper()} (baseline)"
        fig.add_trace(go.Scatter(
            x=s["shot_label"], y=s["test_roc_auc"],
            mode="lines+markers", name=name,
            legendgroup=name, showlegend=name not in seen,
            line=dict(color=colors.get(tt), dash="dash"), opacity=0.5,
            customdata=s[["display_name", "pretrained_model_run", 'steps']].values,
            hovertemplate="<b>%{customdata[1]}</b><br>shots=%{x}<br>AUC=%{y:.3f}<br>train_steps=%{customdata[2]}<br>%{customdata[0]}<extra></extra>",
        ), row=1, col=col)
        seen.add(name)

    fig.update_xaxes(title_text="Shots", row=1, col=col)

fig.update_yaxes(title_text="Test ROC-AUC", row=1, col=1)
fig.update_layout(
    title=f"{TRAIN_DATASET} → {EVAL_DATASET}",
    height=450, width=1100,
    legend=dict(orientation="h", y=-0.2, x=0.5, xanchor="center"),
    hovermode="closest",
)
fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

TRAIN_DATASET = 'midterm'
EVAL_DATASET = 'midterm'
EVAL_TASKS = ["nm", "lp", "pl"]
colors = {"nm": "#00b4d8", "lp": "#f72585", "pl": "#80b918"}

i_cross = df.train1_dataset.eq(TRAIN_DATASET) & df.eval1_dataset.eq(EVAL_DATASET)
plot_df_cross = df[i_cross].copy()
i_same = (df.train1_dataset.eq(EVAL_DATASET) & df.eval1_dataset.eq(EVAL_DATASET)
          & df.prefix.str.startswith("trained"))
plot_df_same = df[i_same].copy()

fig = make_subplots(rows=1, cols=len(EVAL_TASKS), shared_yaxes=True,
                    subplot_titles=[f"Eval: {t.upper()}" for t in EVAL_TASKS])

seen = set()  # dedupe legend across subplots
for col, eval_task in enumerate(EVAL_TASKS, start=1):
    # Cross (solid)
    sub_c = plot_df_cross[plot_df_cross["eval1_task"] == eval_task]
    for tt in sorted(sub_c["train1_task"].dropna().unique()):
        s = sub_c[sub_c["train1_task"] == tt].sort_values("shot_label").dropna(subset=["test_roc_auc"])
        s.loc[s.test_roc_auc.lt(0.5), 'test_roc_auc'] = 0.5 + (0.5-s.test_roc_auc)
        if not s.pretrained_model_run.duplicated(keep=False).all():
            print(f"Warning: pretrained_model_run is not unique for train1_task={tt} and eval1_task={eval_task}")
        name = tt.upper()
        fig.add_trace(go.Scatter(
            x=s["shot_label"], y=s["test_roc_auc"],
            mode="lines+markers", name=name,
            legendgroup=name, showlegend=name not in seen,
            line=dict(color=colors.get(tt)),
            customdata=s[["display_name", "pretrained_model_run", 'steps']].values,
            hovertemplate="<b>%{customdata[1]}</b><br>shots=%{x}<br>AUC=%{y:.3f}<br>train_steps=%{customdata[2]}<br>%{customdata[0]}<extra></extra>",
        ), row=1, col=col)
        seen.add(name)

    # Same-dataset baseline (dashed)
    sub_s = plot_df_same[plot_df_same["eval1_task"] == eval_task]
    for tt in sorted(sub_s["train1_task"].dropna().unique()):
        s = sub_s[sub_s["train1_task"] == tt].sort_values("shot_label").dropna(subset=["test_roc_auc"])
        if not s.pretrained_model_run.duplicated(keep=False).all():
            print(f"Warning: pretrained_model_run is not unique for train1_task={tt} and eval1_task={eval_task}")
        name = f"{tt.upper()} (baseline)"
        fig.add_trace(go.Scatter(
            x=s["shot_label"], y=s["test_roc_auc"],
            mode="lines+markers", name=name,
            legendgroup=name, showlegend=name not in seen,
            line=dict(color=colors.get(tt), dash="dash"), opacity=0.5,
            customdata=s[["display_name", "pretrained_model_run", 'steps']].values,
            hovertemplate="<b>%{customdata[1]}</b><br>shots=%{x}<br>AUC=%{y:.3f}<br>train_steps=%{customdata[2]}<br>%{customdata[0]}<extra></extra>",
        ), row=1, col=col)
        seen.add(name)

    fig.update_xaxes(title_text="Shots", row=1, col=col)

fig.update_yaxes(title_text="Test ROC-AUC", row=1, col=1)
fig.update_layout(
    title=f"{TRAIN_DATASET} → {EVAL_DATASET}",
    height=450, width=1100,
    legend=dict(orientation="h", y=-0.2, x=0.5, xanchor="center"),
    hovermode="closest",
)
fig.show()